# 用 LangGraph 编排人工介入：对账差异审批

本例把“检查差异 → 生成建议 → 等待人工 → 执行或取消”编排成一个图。
重点是 `StateGraph`、`interrupt()`、检查点和 `Command(resume=...)`，不调用大模型，无需 API Key。
金额统一使用整数“分”，所有执行均为模拟，不修改真实账务。

**操作方式**：从上到下运行。在第 5 节“人工回复”单元格停下，查看待审批信息并填写回复，再运行恢复单元格。
“全部运行”会使用预设回复完成演示，因此不要求终端 `input()`，也不会阻塞 Notebook。

```text
START → check_difference ── 无差异 ──→ END
             │ 有差异
             ↓
       propose_adjustment
             ↓
        human_review ←── 非法回复，重新等待
        interrupt()      
          │      │
     批准/修改   拒绝
          ↓      ↓
       execute  cancel
          └──→ END ←──┘
```

## 1. 环境

推荐 Python 3.13，版本与本项目当前依赖对齐。若当前内核缺少依赖，取消下面安装命令的注释并运行，然后重启内核。
`%pip` 会安装到当前 Notebook 内核对应的环境。

In [1]:
# %pip install "langgraph==1.2.11" "langgraph-checkpoint-sqlite==3.1.1"

from importlib.metadata import version
print("LangGraph:", version("langgraph"))
print("SQLite checkpointer:", version("langgraph-checkpoint-sqlite"))

LangGraph: 1.2.11
SQLite checkpointer: 3.1.1


## 2. 定义共享状态和普通节点

节点只返回需要更新的字段；LangGraph 将它们合并进共享状态。
`audit_log` 使用 `operator.add` 追加日志，其余字段默认覆盖。
`difference_cents = 银行金额 - 内部金额`：负数表示建议下调内部金额。

In [2]:
import json
import operator
from typing import Annotated, Literal, TypedDict
from uuid import uuid4

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class ReconciliationState(TypedDict, total=False):
    order_id: str
    internal_amount_cents: int
    bank_amount_cents: int
    difference_cents: int
    proposal: dict
    decision: str
    reviewer: str
    review_note: str
    review_error: str
    status: str
    execution: dict
    audit_log: Annotated[list[str], operator.add]


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


def new_config(label):
    # 业务系统应持久保存此 ID；示例用 UUID 避免反复运行单元格时串用旧状态。
    return {"configurable": {"thread_id": f"{label}-{uuid4().hex}"}}


def check_difference(state):
    for key in ("internal_amount_cents", "bank_amount_cents"):
        if type(state[key]) is not int or state[key] < 0:
            raise ValueError(f"{key} 必须是非负整数（分）")
    difference = state["bank_amount_cents"] - state["internal_amount_cents"]
    return {
        "difference_cents": difference,
        "status": "matched" if difference == 0 else "difference_found",
        "audit_log": [f"检查差异：{difference} 分"],
    }


def route_difference(state) -> Literal["propose_adjustment", "__end__"]:
    return "propose_adjustment" if state["difference_cents"] else END


def propose_adjustment(state):
    # 用确定性规则模拟建议生成；接入 LLM 时可替换此节点，审批机制不变。
    return {
        "proposal": {
            "adjustment_cents": state["difference_cents"],
            "reason": "按银行金额修正内部金额，等待人工核实差异原因",
        },
        "status": "pending_review",
        "audit_log": ["已生成调整建议"],
    }


def execute_adjustment(state):
    # 只计算模拟结果，无真实数据库写入或外部调用。
    adjustment = state["proposal"]["adjustment_cents"]
    new_amount = state["internal_amount_cents"] + adjustment
    remaining = state["bank_amount_cents"] - new_amount
    return {
        "status": "simulated_executed",
        "execution": {
            "adjustment_cents": adjustment,
            "new_internal_amount_cents": new_amount,
            "remaining_difference_cents": remaining,
            "reconciled": remaining == 0,
            "reason": state["proposal"]["reason"],
        },
        "audit_log": [f"模拟执行：调整 {adjustment} 分，剩余差异 {remaining} 分"],
    }


def cancel_adjustment(state):
    return {"status": "rejected", "audit_log": ["人工拒绝，取消调整"]}

## 3. 人工介入节点

第一次调用 `interrupt(payload)` 时，图保存检查点并将待审内容返回给调用方；执行节点尚未运行。
之后传入的 `Command(resume=回复字典)` 会成为 `interrupt()` 的返回值。

本例支持 `approve`、`edit`、`reject`。修改金额必须保持原差异方向且不能超过原差异的绝对值。
允许只处理部分差异，所以“执行完成”不一定代表“对账一致”，请检查 `remaining_difference_cents`。
非法回复写入错误提示后，通过图的自循环再次暂停。

**恢复时整个审批节点会从头执行**，因此 `interrupt()` 前只构造数据，不做写库等副作用；也不要用宽泛的 `try/except` 包住 `interrupt()`。

In [3]:
def validate_reply(reply, state):
    if not isinstance(reply, dict):
        raise ValueError("回复必须是字典")
    if reply.get("action") not in ("approve", "edit", "reject"):
        raise ValueError("action 必须是 approve / edit / reject")
    for key in ("reviewer", "note"):
        if not isinstance(reply.get(key), str) or not reply[key].strip():
            raise ValueError(f"{key} 必须是非空字符串")

    proposal = dict(state["proposal"])
    if reply["action"] == "edit":
        edited = reply.get("edited_proposal")
        if not isinstance(edited, dict):
            raise ValueError("edit 需要 edited_proposal 字典")
        amount = edited.get("adjustment_cents")
        difference = state["difference_cents"]
        if (type(amount) is not int or amount == 0
                or not min(0, difference) <= amount <= max(0, difference)):
            raise ValueError("修改金额必须是非零整数，方向与原差异一致且不超出原差异")
        if not isinstance(edited.get("reason"), str) or not edited["reason"].strip():
            raise ValueError("修改建议需要非空 reason")
        proposal = {"adjustment_cents": amount, "reason": edited["reason"].strip()}
    return proposal


def human_review(state) -> Command[Literal["execute", "cancel", "human_review"]]:
    reply = interrupt({
        "question": "请核实对账差异，然后批准、修改后批准或拒绝",
        "order_id": state["order_id"],
        "internal_amount_cents": state["internal_amount_cents"],
        "bank_amount_cents": state["bank_amount_cents"],
        "difference_cents": state["difference_cents"],
        "proposal": state["proposal"],
        "allowed_actions": ["approve", "edit", "reject"],
        "validation_error": state.get("review_error", ""),
    })

    # 校验在 interrupt 返回之后进行，避免捕获 LangGraph 自己的中断信号。
    try:
        proposal = validate_reply(reply, state)
    except ValueError as exc:
        return Command(
            update={"review_error": str(exc)},
            goto="human_review",
        )

    return Command(
        update={
            "proposal": proposal,
            "decision": reply["action"],
            "reviewer": reply["reviewer"].strip(),
            "review_note": reply["note"].strip(),
            "review_error": "",
            "audit_log": [f"人工审批：{reply['action']}，审批人：{reply['reviewer']}"],
        },
        goto="cancel" if reply["action"] == "reject" else "execute",
    )

## 4. 编排与编译

`checkpointer` 保存进度；`thread_id` 标识同一业务任务。
审批节点通过返回 `Command(goto=...)` 选择下一节点，因此这里不再给它添加普通出边。
这里使用 `invoke()` 方便观察完整返回值，不需要额外流式接口。

In [4]:
def build_graph(checkpointer):
    builder = StateGraph(ReconciliationState)
    builder.add_node("check_difference", check_difference)
    builder.add_node("propose_adjustment", propose_adjustment)
    builder.add_node("human_review", human_review)
    builder.add_node("execute", execute_adjustment)
    builder.add_node("cancel", cancel_adjustment)
    builder.add_edge(START, "check_difference")
    builder.add_conditional_edges("check_difference", route_difference)
    builder.add_edge("propose_adjustment", "human_review")
    builder.add_edge("execute", END)
    builder.add_edge("cancel", END)
    return builder.compile(checkpointer=checkpointer)


graph = build_graph(InMemorySaver())
sample = {
    "order_id": "ORDER-DEMO-001",
    "internal_amount_cents": 100_000,  # 1000.00 元
    "bank_amount_cents": 99_500,      # 995.00 元，差异 -500 分
    "audit_log": [],
}

## 5. 分开运行：暂停 → 查看 → 人工回复 → 恢复

先运行下面单元格。预期状态为 `pending_review`，下一节点仍是 `human_review`，没有 `execution`。
`invoke()` 此时已经返回，审批期间不占着一个等待输入的调用。

In [5]:
config = new_config("manual-edit")
paused = graph.invoke(sample, config=config)
assert "__interrupt__" in paused
assert paused["status"] == "pending_review"
assert "execution" not in paused
show(paused["__interrupt__"][0].value)
print("保存的待执行节点：", graph.get_state(config).next)

{
  "question": "请核实对账差异，然后批准、修改后批准或拒绝",
  "order_id": "ORDER-DEMO-001",
  "internal_amount_cents": 100000,
  "bank_amount_cents": 99500,
  "difference_cents": -500,
  "proposal": {
    "adjustment_cents": -500,
    "reason": "按银行金额修正内部金额，等待人工核实差异原因"
  },
  "allowed_actions": [
    "approve",
    "edit",
    "reject"
  ],
  "validation_error": ""
}
保存的待执行节点： ('human_review',)


### 填写人工回复

修改下面字典，然后运行下一单元格恢复。本例确认其中 4.50 元为手续费，剩余 0.50 元留待核查。
把 `action` 改为 `approve` 可批准原建议，改为 `reject` 可拒绝；这两种动作不读取 `edited_proposal`。
每次重新体验，请先运行上一个“暂停”单元格，创建新任务；不要重复恢复已结束的任务。

In [6]:
human_reply = {
    "action": "edit",
    "reviewer": "学习者",
    "note": "已核实手续费，只批准已确认的部分差异",
    "edited_proposal": {
        "adjustment_cents": -450,
        "reason": "银行手续费 4.50 元，剩余 0.50 元另行核查",
    },
}

In [7]:
# 关键：继续使用暂停时的 config，而不是生成新的 thread_id。
final = graph.invoke(Command(resume=human_reply), config=config)
if "__interrupt__" in final:
    print("回复校验未通过，请修改 human_reply 后再次运行本单元格：")
    show(final["__interrupt__"][0].value)
else:
    show(final)
    print("任务已结束：", not graph.get_state(config).next)

{
  "order_id": "ORDER-DEMO-001",
  "internal_amount_cents": 100000,
  "bank_amount_cents": 99500,
  "difference_cents": -500,
  "proposal": {
    "adjustment_cents": -450,
    "reason": "银行手续费 4.50 元，剩余 0.50 元另行核查"
  },
  "decision": "edit",
  "reviewer": "学习者",
  "review_note": "已核实手续费，只批准已确认的部分差异",
  "review_error": "",
  "status": "simulated_executed",
  "execution": {
    "adjustment_cents": -450,
    "new_internal_amount_cents": 99550,
    "remaining_difference_cents": -50,
    "reconciled": false,
    "reason": "银行手续费 4.50 元，剩余 0.50 元另行核查"
  },
  "audit_log": [
    "检查差异：-500 分",
    "已生成调整建议",
    "人工审批：edit，审批人：学习者",
    "模拟执行：调整 -450 分，剩余差异 -50 分"
  ]
}
任务已结束： True


## 6. 验证分支、输入校验和任务隔离

以下用固定回复模拟人工决定，便于“全部运行”时验证所有路径；不会替代上面的手动审批体验。
同时验证：暂停时未执行、拒绝后未执行、修改确实影响结果、非法回复仍等待，以及其他任务不会被一起恢复。

In [8]:
approve_reply = {"action": "approve", "reviewer": "复核员", "note": "确认全部差异"}
reject_reply = {"action": "reject", "reviewer": "复核员", "note": "凭证不足，暂不调整"}
edit_reply = {
    "action": "edit", "reviewer": "复核员", "note": "仅处理已核实部分",
    "edited_proposal": {"adjustment_cents": -450, "reason": "已核实手续费"},
}

for reply in (approve_reply, edit_reply, reject_reply):
    case_config = new_config(reply["action"])
    waiting = graph.invoke(sample, config=case_config)
    assert "__interrupt__" in waiting and "execution" not in waiting
    result = graph.invoke(Command(resume=reply), config=case_config)
    assert "__interrupt__" not in result and not graph.get_state(case_config).next
    # 恢复不会重跑已经完成的上游节点。
    assert result["audit_log"].count("已生成调整建议") == 1
    if reply["action"] == "reject":
        assert result["status"] == "rejected" and "execution" not in result
    else:
        expected = -450 if reply["action"] == "edit" else -500
        assert result["execution"]["adjustment_cents"] == expected
        assert result["execution"]["remaining_difference_cents"] == -500 - expected
        assert result["execution"]["reconciled"] == (expected == -500)
    print(reply["action"], "→", result["status"])

matched = graph.invoke(
    {**sample, "bank_amount_cents": 100_000}, config=new_config("matched")
)
assert matched["status"] == "matched"
assert "__interrupt__" not in matched and "execution" not in matched
print("无差异 →", matched["status"])

config_a, config_b = new_config("task-a"), new_config("task-b")
graph.invoke(sample, config=config_a)
graph.invoke({**sample, "order_id": "ORDER-DEMO-002"}, config=config_b)
invalid = graph.invoke(
    Command(resume={**edit_reply, "edited_proposal": {"adjustment_cents": 100, "reason": "方向错误"}}),
    config=config_a,
)
assert "__interrupt__" in invalid and "execution" not in invalid
assert invalid["__interrupt__"][0].value["validation_error"]
print("非法回复 → 再次暂停：", invalid["__interrupt__"][0].value["validation_error"])
corrected = graph.invoke(Command(resume=approve_reply), config=config_a)
assert corrected["execution"]["reconciled"]
assert graph.get_state(config_b).next == ("human_review",)
assert "execution" not in graph.get_state(config_b).values
graph.invoke(Command(resume=reject_reply), config=config_b)
print("输入纠正、任务隔离检查通过")

approve → simulated_executed
edit → simulated_executed
reject → rejected
无差异 → matched
非法回复 → 再次暂停： 修改金额必须是非零整数，方向与原差异一致且不超出原差异
输入纠正、任务隔离检查通过


## 7. SQLite：关闭连接、重建图，再恢复审批

`InMemorySaver` 的数据只活在当前进程中。下面改用 `SqliteSaver`，两个独立的 `with` 分别代表审批前后两次打开数据库。
第二次只读取已保存的状态，不重新提交原始业务输入。
这验证的是关闭连接并重建图后的恢复；真实重启应用时也需重新定义节点、连接同一个文件，并取回同一个 `thread_id`。

为了反复运行不会污染仓库，每次示例将数据库建在系统临时目录并打印路径；文件保留以供检查。
若要跨 Notebook 内核重启手动恢复，请记录打印出的数据库路径和 `thread_id`，重新执行节点定义后用它们构建配置。

In [9]:
from pathlib import Path
from tempfile import mkdtemp
from langgraph.checkpoint.sqlite import SqliteSaver

db_path = Path(mkdtemp(prefix="langgraph-hitl-")) / "checkpoints.sqlite"
durable_config = new_config("sqlite-review")
print("数据库：", db_path)
print("thread_id：", durable_config["configurable"]["thread_id"])

with SqliteSaver.from_conn_string(str(db_path)) as saver_before:
    graph_before = build_graph(saver_before)
    durable_paused = graph_before.invoke(sample, config=durable_config)
    assert "__interrupt__" in durable_paused
    assert "execution" not in durable_paused
print("第一次连接已关闭，审批仍保存在 SQLite 中")

数据库： C:\Users\Woda\AppData\Local\Temp\langgraph-hitl-dh2a6c72\checkpoints.sqlite
thread_id： sqlite-review-30d5e051285d409ab818325fabfb2f06
第一次连接已关闭，审批仍保存在 SQLite 中


In [10]:
with SqliteSaver.from_conn_string(str(db_path)) as saver_after:
    graph_after = build_graph(saver_after)
    saved = graph_after.get_state(durable_config)
    assert saved.next == ("human_review",)
    assert saved.tasks[0].interrupts
    print("从检查点读取待审批内容：")
    show(saved.tasks[0].interrupts[0].value)
    durable_final = graph_after.invoke(
        Command(resume=approve_reply), config=durable_config
    )
    assert durable_final["execution"]["reconciled"]
    assert not graph_after.get_state(durable_config).next
    print("重建图后恢复成功：", durable_final["status"])

从检查点读取待审批内容：
{
  "question": "请核实对账差异，然后批准、修改后批准或拒绝",
  "order_id": "ORDER-DEMO-001",
  "internal_amount_cents": 100000,
  "bank_amount_cents": 99500,
  "difference_cents": -500,
  "proposal": {
    "adjustment_cents": -500,
    "reason": "按银行金额修正内部金额，等待人工核实差异原因"
  },
  "allowed_actions": [
    "approve",
    "edit",
    "reject"
  ],
  "validation_error": ""
}
重建图后恢复成功： simulated_executed


## 8. 接到真实应用时的数据流

1. 创建任务的接口调用 `graph.invoke(业务输入, config)`，把 `__interrupt__` 中的内容返回审批页面，同时保存任务对应的 `thread_id`。
2. 审批页面收集操作者回复；审批接口在验证操作者权限后，以相同检查点存储和相同 `thread_id` 调用 `graph.invoke(Command(resume=回复), config)`。
3. 如果结果仍含 `__interrupt__`，继续展示待审信息；否则展示最终状态。图可以结束但仍有剩余差异，业务上需另建后续核查任务。

本例的 `reviewer` 是学习用输入，真实应用应从登录身份获取。真实执行节点还需按业务任务 ID 做幂等，检查点本身不保证外部写入恰好一次。
同步 `SqliteSaver` 适合此本地学习示例；异步接口应使用相应异步检查点与 `ainvoke()`。
本例手动校验回复，兼容 `langgraph==1.2.11`，没有使用更高版本才提供的 `interrupt(response_schema=...)`。

官方资料：[Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)、[Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)。